# 02 - Module 2: BI analysis with SQL and PySpark

Hands-on companion for [`module-2-bi-with-sql.md`](../module-2-bi-with-sql.md).

**Every KPI is shown three ways:**

1. The **SSIS data flow** you would have built in your previous BI lab
   (text description above each query).
2. The **lakehouse SQL** you run against the serverless warehouse.
3. The **PySpark DataFrame** equivalent.

Run the next cell once to anchor the schema for both languages.


In [ ]:
%sql
USE workspace.bi_course;


In [ ]:
spark.sql("USE workspace.bi_course")


---

## KPI 1 - Revenue by product

**SSIS design:** OLE DB Source `fact_sales` -> Lookup on `dim_product` ->
Aggregate (`SUM(revenue)` by `product_name`) -> Sort desc -> Destination.


In [ ]:
%sql
SELECT
    p.product_name,
    SUM(f.revenue) AS total_revenue,
    SUM(f.quantity) AS units_sold
FROM fact_sales        f
JOIN dim_product       p USING (product_id)
GROUP BY p.product_name
ORDER BY total_revenue DESC;


In [ ]:
from pyspark.sql import functions as F

fact    = spark.table("fact_sales")
product = spark.table("dim_product")

kpi1 = (
    fact.join(product, on="product_id", how="inner")
        .groupBy("product_name")
        .agg(
            F.sum("revenue").alias("total_revenue"),
            F.sum("quantity").alias("units_sold"),
        )
        .orderBy(F.col("total_revenue").desc())
)
kpi1.show()


---

## KPI 2 - Daily revenue + day-over-day delta

**SSIS design:** OLE DB Source -> Aggregate by `order_date` -> Sort asc
-> Script Component (or staged self-join) for previous-day revenue.


In [ ]:
%sql
WITH daily AS (
    SELECT
        order_date,
        SUM(revenue) AS daily_revenue
    FROM fact_sales
    GROUP BY order_date
)
SELECT
    order_date,
    daily_revenue,
    LAG(daily_revenue) OVER (ORDER BY order_date) AS prev_day_revenue,
    daily_revenue
        - COALESCE(LAG(daily_revenue) OVER (ORDER BY order_date), 0)
        AS dod_delta
FROM daily
ORDER BY order_date;


In [ ]:
from pyspark.sql import Window

w = Window.orderBy("order_date")

daily = (
    spark.table("fact_sales")
         .groupBy("order_date")
         .agg(F.sum("revenue").alias("daily_revenue"))
)

kpi2 = (
    daily.withColumn("prev_day_revenue", F.lag("daily_revenue").over(w))
         .withColumn(
             "dod_delta",
             F.col("daily_revenue") - F.coalesce(F.col("prev_day_revenue"), F.lit(0)),
         )
         .orderBy("order_date")
)
kpi2.show()


---

## KPI 3 - Region x product matrix with subtotals (`ROLLUP`)

**SSIS design:** Source -> two Lookups -> Multicast -> two parallel
Aggregate branches `(region, product)` and `(region)` -> Union All with
manual `NULL` markers for subtotal rows.


In [ ]:
%sql
SELECT
    COALESCE(r.region_name,  '-- Grand Total --') AS region,
    COALESCE(p.product_name, '* All products *')  AS product,
    SUM(f.revenue) AS revenue
FROM fact_sales        f
JOIN dim_region        r USING (region_id)
JOIN dim_product       p USING (product_id)
GROUP BY ROLLUP (r.region_name, p.product_name)
ORDER BY r.region_name NULLS LAST,
         p.product_name NULLS LAST;


In [ ]:
fact    = spark.table("fact_sales")
product = spark.table("dim_product")
region  = spark.table("dim_region")

kpi3 = (
    fact.join(region,  on="region_id",  how="inner")
        .join(product, on="product_id", how="inner")
        .rollup("region_name", "product_name")
        .agg(F.sum("revenue").alias("revenue"))
        .orderBy("region_name", "product_name")
)
kpi3.show(50)


---

## KPI 4 - Top-2 products per region (window function)

**SSIS design:** Sort per region partition -> Script Component (or staged
`ROW_NUMBER()` query) -> Conditional Split on `rank <= 2`. Several
hundred picas of canvas in SSIS; one CTE here.


In [ ]:
%sql
WITH ranked AS (
    SELECT
        r.region_name,
        p.product_name,
        SUM(f.revenue) AS product_revenue,
        ROW_NUMBER() OVER (
            PARTITION BY r.region_name
            ORDER BY     SUM(f.revenue) DESC
        ) AS rn
    FROM fact_sales       f
    JOIN dim_region       r USING (region_id)
    JOIN dim_product      p USING (product_id)
    GROUP BY r.region_name, p.product_name
)
SELECT region_name, product_name, product_revenue
FROM   ranked
WHERE  rn <= 2
ORDER BY region_name, product_revenue DESC;


In [ ]:
agg = (
    spark.table("fact_sales")
         .join(spark.table("dim_region"),  on="region_id",  how="inner")
         .join(spark.table("dim_product"), on="product_id", how="inner")
         .groupBy("region_name", "product_name")
         .agg(F.sum("revenue").alias("product_revenue"))
)

w_top = Window.partitionBy("region_name").orderBy(F.col("product_revenue").desc())

kpi4 = (
    agg.withColumn("rn", F.row_number().over(w_top))
       .filter("rn <= 2")
       .orderBy("region_name", F.col("product_revenue").desc())
       .drop("rn")
)
kpi4.show()


Module 2 complete. Open `03_gold_views.ipynb` to materialise these
KPIs as governed views that the AI/BI Dashboard will consume.
